In [ ]:
import json
import random

import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

import config as cfg
from config import configure_mlflow_tracking

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential

# ==================================================
# Configuration
# ==================================================

In [ ]:
DATA_PATH = cfg.REPORT_DIR / "all_features_dataset.csv"
MODEL_PATH = cfg.MODEL_DIR / "lstm_model.keras"
PLOT_DIR = cfg.REPORT_DIR / "plots"

WINDOW_SIZE = 24 * 7
FORECAST_HORIZON = 72

TRAIN_END_DATE = "2024-04-30"
VALIDATION_START_DATE = "2024-05-01"
VALIDATION_END_DATE = "2025-04-30"
TEST_START_DATE = "2025-05-01"
TEST_END_DATE = "2026-04-30"

EPOCHS = 50
BATCH_SIZE = 64
DROPOUT_RATE = 0.2
EARLY_STOPPING_PATIENCE = 5
  
PERMUTATION_SAMPLE_SIZE = 500
RUN_PERMUTATION_IMPORTANCE = False

TARGET_COL = "Demand"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
FEATURES = [
    "Demand",
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "wind_speed_10m",
    "snowfall",
    "precipitation",
    "cloud_cover",
    "holiday_encoded",

    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "month_sin",
    "month_cos",

    "demand_lag_1h",
    "demand_lag_2h",
    "demand_lag_3h",
    "demand_lag_4h",
    "demand_lag_24h",
    "demand_lag_48h",
    "demand_lag_72h",

    "demand_rolling_24h_mean",
    "demand_rolling_48h_mean",
    "demand_rolling_72h_mean",
    "demand_rolling_168h_mean",

    "demand_std_24h",
    "demand_std_48h",
    "demand_std_72h",
    "demand_std_168h",

    "demand_min_24h",
    "demand_min_48h",
    "demand_min_72h",
    "demand_min_168h",

    "demand_max_24h",
    "demand_max_48h",
    "demand_max_72h",
    "demand_max_168h",

    "NYNGSP",
    "NYPOP",
]

# ==================================================
# Subfunctions
# ==================================================

In [ ]:
def inverse_transform_target_sequences(scaler, sequences):
    reshaped = np.asarray(sequences).reshape(-1, 1)

    dummy = np.zeros((reshaped.shape[0], len(FEATURES)))
    dummy[:, 0] = reshaped[:, 0]

    restored = scaler.inverse_transform(dummy)[:, 0]

    return restored.reshape(np.asarray(sequences).shape)

In [ ]:
def create_sequences(scaled_data):
    min_required_rows = WINDOW_SIZE + FORECAST_HORIZON

    if len(scaled_data) < min_required_rows:
        raise ValueError(
            "Not enough rows to create sequences for the current "
            f"WINDOW_SIZE={WINDOW_SIZE} and FORECAST_HORIZON={FORECAST_HORIZON}. "
            f"Need at least {min_required_rows} rows, got {len(scaled_data)}."
        )

    X = []
    y = []

    for i in range(WINDOW_SIZE, len(scaled_data) - FORECAST_HORIZON + 1):
        X.append(scaled_data[i - WINDOW_SIZE:i])
        y.append(scaled_data[i:i + FORECAST_HORIZON, 0])

    return np.array(X), np.array(y)

In [ ]:
def build_lstm_model():
    model = Sequential(
        [
            Input(shape=(WINDOW_SIZE, len(FEATURES))),

            LSTM(
                units=128,
                return_sequences=True,
            ),
            Dropout(DROPOUT_RATE),

            LSTM(
                units=64,
                return_sequences=False,
            ),
            Dropout(DROPOUT_RATE),

            Dense(FORECAST_HORIZON),
        ]
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=0.001,
        clipnorm=1.0,
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
    )

    return model

In [ ]:
def save_training_loss_plot(history, output_path):
    plt.figure(figsize=(8, 5))
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.legend()
    plt.title("LSTM Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

In [ ]:
def save_forecast_plot(y_test_flat, y_pred_flat, output_path, limit=500):
    plt.figure(figsize=(15, 5))
    plt.plot(y_test_flat[:limit], label="Actual")
    plt.plot(y_pred_flat[:limit], label="Forecast")
    plt.legend()
    plt.title("Actual vs Forecast on Test Set")
    plt.xlabel("Forecasted point")
    plt.ylabel(TARGET_COL)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

In [ ]:
def save_permutation_importance_plot(importance_df, output_path):
    plt.figure(figsize=(10, 10))

    plt.barh(
        importance_df["feature"],
        importance_df["importance"],
    )

    plt.xlabel("Increase in MAE")
    plt.title("Permutation Importance")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

In [ ]:
def calculate_mape(y_true, y_pred, epsilon=1e-6):
    nonzero_mask = np.abs(y_true) > epsilon
    excluded_zeros = np.size(y_true) - np.count_nonzero(nonzero_mask)

    if np.any(nonzero_mask):
        mape = (
            np.mean(
                np.abs(
                    (y_true[nonzero_mask] - y_pred[nonzero_mask])
                    / y_true[nonzero_mask]
                )
            )
            * 100
        )
    else:
        mape = np.nan

    return mape, excluded_zeros

In [ ]:
def evaluate_mae(model, X, y_true, scaler, batch_size=64):
    y_pred = model.predict(
        X,
        batch_size=batch_size,
        verbose=0,
    )

    y_pred_real = inverse_transform_target_sequences(
        scaler,
        y_pred,
    )

    y_true_real = inverse_transform_target_sequences(
        scaler,
        y_true,
    )

    mae = mean_absolute_error(
        y_true_real.flatten(),
        y_pred_real.flatten(),
    )

    return mae

In [ ]:
def permutation_importance(
    model,
    X_test,
    y_test,
    scaler,
    feature_names,
    sample_size=500,
    batch_size=64,
    random_state=42,
):
    rng = np.random.default_rng(random_state)

    n_samples = min(sample_size, len(X_test))

    sample_idx = rng.choice(
        len(X_test),
        size=n_samples,
        replace=False,
    )

    X_sample = X_test[sample_idx].copy()
    y_sample = y_test[sample_idx].copy()

    baseline_mae = evaluate_mae(
        model=model,
        X=X_sample,
        y_true=y_sample,
        scaler=scaler,
        batch_size=batch_size,
    )

    print("\n===== PERMUTATION IMPORTANCE =====")
    print(f"Sample size: {n_samples}")
    print(f"Baseline MAE on sampled test set: {baseline_mae:.3f}")
    print("-" * 60)

    results = []

    for feature_idx, feature_name in enumerate(feature_names):
        X_perm = X_sample.copy()

        perm_idx = rng.permutation(n_samples)

        X_perm[:, :, feature_idx] = X_perm[perm_idx, :, feature_idx]

        permuted_mae = evaluate_mae(
            model=model,
            X=X_perm,
            y_true=y_sample,
            scaler=scaler,
            batch_size=batch_size,
        )

        importance = permuted_mae - baseline_mae

        results.append(
            {
                "feature": feature_name,
                "baseline_mae": baseline_mae,
                "permuted_mae": permuted_mae,
                "importance": importance,
            }
        )

        print(
            f"{feature_idx + 1:02d}/{len(feature_names)} "
            f"{feature_name:35s} "
            f"importance = {importance:.3f}"
        )

    importance_df = (
        pd.DataFrame(results)
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

    return importance_df

In [ ]:
def log_mlflow_run_metadata(df, train_df, val_df, test_df, X_train, X_val, X_test):
    params = {
        "data_path": str(DATA_PATH),
        "target_col": TARGET_COL,

        "window_size": WINDOW_SIZE,
        "forecast_horizon": FORECAST_HORIZON,

        "train_end_date_config": TRAIN_END_DATE,
        "validation_start_date_config": VALIDATION_START_DATE,
        "validation_end_date_config": VALIDATION_END_DATE,
        "test_start_date_config": TEST_START_DATE,
        "test_end_date_config": TEST_END_DATE,

        "train_start_date_actual": str(train_df["date"].min()),
        "train_end_date_actual": str(train_df["date"].max()),
        "validation_start_date_actual": str(val_df["date"].min()),
        "validation_end_date_actual": str(val_df["date"].max()),
        "test_start_date_actual": str(test_df["date"].min()),
        "test_end_date_actual": str(test_df["date"].max()),

        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "dropout_rate": DROPOUT_RATE,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,

        "optimizer": "adam",
        "loss": "mse",

        "feature_count": len(FEATURES),
        "row_count_after_dropna": len(df),
        "train_row_count": len(train_df),
        "validation_row_count": len(val_df),
        "test_row_count": len(test_df),

        "train_sequence_count": len(X_train),
        "validation_sequence_count": len(X_val),
        "test_sequence_count": len(X_test),

        "seed": SEED,
        "lstm_architecture": "Input_LSTM128_LSTM64_Dense",

        "reduce_lr_on_plateau": True,
        "reduce_lr_factor": 0.5,
        "reduce_lr_patience": 3,

        "scaler_fit_on": "train_only",
        "validation_history_context_hours": WINDOW_SIZE,
        "test_history_context_hours": WINDOW_SIZE,

        "permutation_importance_sample_size": PERMUTATION_SAMPLE_SIZE,
    }

    mlflow.log_params(params)
    mlflow.log_text(json.dumps(FEATURES, indent=2), "features.json")

# ==================================================
# Train model
# ==================================================

In [ ]:
configure_mlflow_tracking()

# Loading data and Train/Validation/Test Split

In [ ]:
print("Loading data...")
df = pd.read_csv(DATA_PATH)

missing_raw_columns = [TARGET_COL, "date"]
missing_raw_columns = [col for col in missing_raw_columns if col not in df.columns]

if missing_raw_columns:
    raise KeyError(f"Missing required columns in dataset: {missing_raw_columns}")

df[TARGET_COL] = pd.to_numeric(
    df[TARGET_COL].astype(str).str.replace(",", "", regex=False),
    errors="coerce",
    )

df["date"] = pd.to_datetime(df["date"], errors="coerce")

df = df.sort_values("date").reset_index(drop=True)

missing_features = [feature for feature in FEATURES if feature not in df.columns]

if missing_features:
    raise KeyError(f"Missing features in dataset: {missing_features}")

df = df[["date"] + FEATURES].copy()

df = df.dropna().reset_index(drop=True)

print("Data shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())

print("Splitting data by date...")

train_end = pd.to_datetime(TRAIN_END_DATE)
val_start = pd.to_datetime(VALIDATION_START_DATE)
val_end = pd.to_datetime(VALIDATION_END_DATE)
test_start = pd.to_datetime(TEST_START_DATE)
test_end = pd.to_datetime(TEST_END_DATE)

train_df = df[df["date"] <= train_end].copy()

val_df = df[
    (df["date"] >= val_start)
    & (df["date"] <= val_end)
    ].copy()

test_df = df[
    (df["date"] >= test_start)
    & (df["date"] <= test_end)
    ].copy()

if train_df.empty:
    raise ValueError("Training set is empty. Check TRAIN_END_DATE.")

if val_df.empty:
    raise ValueError("Validation set is empty. Check validation date range.")

if test_df.empty:
    raise ValueError("Test set is empty. Check test date range.")

print("=" * 60)
print("TRAIN")
print("Date range:", train_df["date"].min(), "to", train_df["date"].max())
print("Rows:", len(train_df))
print()

print("VALIDATION")
print("Date range:", val_df["date"].min(), "to", val_df["date"].max())
print("Rows:", len(val_df))
print()

print("TEST")
print("Date range:", test_df["date"].min(), "to", test_df["date"].max())
print("Rows:", len(test_df))
print("=" * 60)

# Scaling data..